# 📘 Module 3.4 — Core Prompting Techniques
### Agentic AI Engineer Roadmap · Personal Study Notebook

---

> **Mentor role:** Senior AI Architect & Agentic AI Engineer  
> **Your level:** Python fundamentals ✅ | Agentic AI beginner → production  
> **Goal:** Conceptual understanding + coding knowledge + production intuition

---

## 📋 Table of Contents

| # | Topic | Section |
|---|-------|---------|
| 1 | Zero-shot Prompting | [Jump](#1-zero-shot-prompting) |
| 2 | Few-shot with Curated Examples | [Jump](#2-few-shot-with-curated-examples) |
| 3 | COSTAR Framework | [Jump](#3-costar-framework) |
| 4 | Iterative Refinement Loop | [Jump](#4-iterative-refinement-loop) |
| 5 | Big Picture Summary | [Jump](#5-big-picture-summary) |
| 6 | Knowledge Check (10 Questions) | [Jump](#6-knowledge-check) |
| 7 | Mini Projects | [Jump](#7-mini-projects) |

---

### 🗺️ How All 4 Techniques Connect

```
COMPLEXITY OF TASK  →  →  →  →  →  →  →  →  →  →  →

Simple           Medium            Complex          Very Complex
   │                │                  │                 │
   ▼                ▼                  ▼                 ▼
Zero-shot       Few-shot           COSTAR         COSTAR +
(Just ask)    (Show pattern)  (Full structure)   Iterative Loop

THESE ARE NOT EXCLUSIVE — in production you COMBINE them:
COSTAR system prompt  +  Few-shot examples  +  Refinement loop
= a fully engineered agentic prompt pipeline
```

## ⚙️ Setup — Install & Import

All code examples in this notebook use the **Anthropic Python SDK**.  
Run this cell once before running any examples.

In [ ]:
# Install the Anthropic SDK (run once)
# !pip install anthropic

import anthropic
import json
import os

# The SDK automatically reads ANTHROPIC_API_KEY from your environment.
# Set it like this (or use a .env file with python-dotenv):
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()

# Model we'll use throughout this notebook
MODEL = "claude-sonnet-4-20250514"

print("✅ Setup complete. Ready to run all examples.")

---
# 1. Zero-shot Prompting
> **Number:** 01 | **Difficulty:** Beginner → Production

---

## 1.1 What Is It?

**Zero-shot prompting** means sending a task to an LLM **without providing any example** of what the output should look like.

- You describe the task in plain language.
- The model uses its training knowledge to figure out what to do.
- **"Zero"** = zero examples. **"Shot"** = example (ML terminology).

---

## 1.2 Why Do We Need It?

| Situation | Why zero-shot works |
|-----------|---------------------|
| Task is straightforward (classify, translate, summarize) | Model already knows these patterns |
| No labeled examples available yet | You have nothing to show |
| Rapid prototyping | Fastest way to test an idea |
| Agent routing decisions | Quick classification inside a pipeline |

---

## 1.3 Real-world Analogy

> 🧑‍💼 **Analogy:** You ask a new employee:
> *"Please write a professional email declining this vendor."*
> You give **no examples** — you trust their existing knowledge.
> That is zero-shot.

---

## 1.4 Agentic AI Context

In real Agentic AI systems, zero-shot is used when an agent needs to:
- **Classify incoming user intent** (route to right sub-agent)
- **Select which tool to call** next
- **Summarize retrieved document chunks** in a RAG pipeline
- **Make quick yes/no decisions** in a workflow graph

```
User message: "Book a flight to Delhi"
        │
        ▼
Agent's Router (Zero-shot prompt)
┌────────────────────────────────┐
│ "Classify this intent into:   │
│  flight / hotel / restaurant" │
└────────────────────────────────┘
        │
        ▼
LLM output: "flight"
        │
        ▼
Agent calls FlightSearchTool
```

## 1.5 Python Example — Basic Zero-shot

In [ ]:
# ZERO-SHOT: No examples, just a clear instruction

prompt = """
Classify the sentiment of the following customer review as:
POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The product arrived late but the quality is excellent."

Respond with only one word.
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=50,
    messages=[{"role": "user", "content": prompt}]
)

result = response.content[0].text.strip()
print(f"Sentiment: {result}")

### 📋 Code Breakdown

| Line / Concept | Explanation |
|----------------|-------------|
| `prompt = """..."""` | Pure zero-shot: task description + input only. **No output examples.** |
| `max_tokens=50` | We only expect 1 word — limiting tokens avoids waste and forces a concise answer |
| `messages=[{"role": "user", ...}]` | The Anthropic SDK takes a list of messages. This is the simplest form. |
| `response.content[0].text` | Extracts the model's text from the API response object |
| `.strip()` | Removes leading/trailing whitespace for clean parsing |

## 1.6 Improved Zero-shot — Adding Role + Format

> 💡 **Intermediate tip:** You can dramatically improve zero-shot without adding examples.  
> Add: (1) a role, (2) explicit output format, (3) step-by-step instruction.

In [ ]:
# IMPROVED ZERO-SHOT: Role + Format specification + Chain-of-thought

system = "You are a senior customer experience analyst. Be precise and data-driven."

prompt = """
Analyze the following customer review. Think step by step.

Review: "The product arrived late but the quality is excellent."

Respond ONLY in this JSON format:
{
  "sentiment": "POSITIVE | NEGATIVE | MIXED | NEUTRAL",
  "issues": ["list of problems mentioned"],
  "positives": ["list of good things mentioned"],
  "confidence": 0.0
}
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    system=system,
    messages=[{"role": "user", "content": prompt}]
)

output = response.content[0].text.strip()
parsed = json.loads(output)

print(json.dumps(parsed, indent=2))

## 1.7 Learning Depth

### 🟢 Beginner View
Zero-shot is just asking the LLM a direct question or giving it a task without showing examples.
Works great for simple, well-known tasks: translation, sentiment classification, summarization.

### 🟡 Intermediate View
Zero-shot quality depends heavily on how well you phrase the instruction.  
Adding a **role**, **output format spec**, and **"think step by step"** can match few-shot performance on many tasks without needing any examples.

### 🔴 Production View
In production agents, zero-shot prompts are:
- Stored in **version-controlled prompt files** (not hardcoded strings)
- Tested against **automated eval sets** before deployment
- Monitored for **accuracy drift** over time
- Upgraded to few-shot or fine-tuning when accuracy drops below a threshold

---

## 1.8 Production Usage

| Framework | Zero-shot Usage |
|-----------|-----------------|
| OpenAI Agents SDK | Intent classification in the agent router step |
| LangGraph | Node-level LLM calls for quick decisions in the graph |
| RAG Pipelines | Summarizing retrieved document chunks |
| CrewAI | Agent role definitions and task descriptions |

---

## 1.9 Common Mistakes

| Mistake | Fix |
|---------|-----|
| ❌ Vague instruction: `"Analyze this text"` | ✅ `"Extract all named organizations from this text as a comma-separated list"` |
| ❌ No output format specified | ✅ Always define the expected format: JSON, one word, bullet list, etc. |
| ❌ Using zero-shot for domain-specific tasks | ✅ Upgrade to few-shot or fine-tuning for niche domains |
| ❌ Not testing edge cases | ✅ Test at least 10–20 varied inputs before deploying |

---

## 1.10 Trade-offs — When NOT to Use Zero-shot

| Aspect | Zero-shot |
|--------|-----------|
| Speed | ✅ Fastest — no examples to engineer |
| Consistency | ⚠️ Variable — model interprets freely |
| Domain tasks | ❌ Poor on niche/domain-specific tasks |
| Output format | ⚠️ Needs explicit format instructions to be reliable |
| **When NOT to use** | Complex multi-step reasoning, domain-specific extraction, tasks needing a specific style you can demonstrate |

---

## 1.11 Interview Questions

**Q1: What is zero-shot prompting and when is it appropriate?**  
A: Zero-shot sends a task to an LLM without any input→output examples. Appropriate when the task is well-defined and common (classification, translation, summarization), when prototyping quickly, or when no labeled examples exist.

**Q2: How do you improve a failing zero-shot prompt without adding examples?**  
A: Add a system-level role, specify exact output format (JSON schema), add chain-of-thought instruction ("Think step by step"), and break complex tasks into smaller sub-instructions.

**Q3: In an agentic system, where exactly is zero-shot typically used?**  
A: Intent routing, quick summarization of tool results, deciding which tool to call next, and generating brief structured JSON for downstream processing.

---
# 2. Few-shot with Curated Examples
> **Number:** 02 | **Difficulty:** Beginner → Production

---

## 2.1 What Is It?

**Few-shot prompting** means including a small number of **input → output examples** inside your prompt before giving the actual task.

- The model learns the **pattern** from your examples and applies it.
- **"Few"** = typically 2–10 examples.
- **"Curated"** = you hand-pick the best, most representative examples — not random ones.

---

## 2.2 Why Do We Need It?

Zero-shot fails when you need:
- A very specific **output format** followed consistently
- **Domain-specific language** or edge case handling
- A particular **style or tone** matched precisely
- A task the model hasn't seen phrased this way before

---

## 2.3 Real-world Analogy

> 🎓 **Analogy:** Training a new customer support agent.  
> Instead of just saying *"respond professionally,"* you show them **3 example tickets** with ideal responses.  
> They instantly understand tone, structure, and phrasing. That's few-shot learning.

---

## 2.4 Agentic AI Context

Few-shot is critical in agents for:

| Agent Type | Few-shot Usage |
|------------|----------------|
| Tool-calling agents | Showing exact JSON tool call formats |
| Data extraction agents | Teaching consistent extraction patterns |
| RAG answer formatters | Ensuring answers follow a specific citation style |
| Code generation agents | Demonstrating coding style conventions |

```
Few-shot prompt structure inside an Agent:

┌─────────────────────────────────────────────────────────┐
│ SYSTEM: You are a data extraction agent.                │
│                                                         │
│ EXAMPLE 1:                                              │
│ Input: "Call John at +91-9876543210 about the deal"     │
│ Output: {"name":"John","phone":"+91-9876543210"}         │
│                                                         │
│ EXAMPLE 2:                                              │
│ Input: "Email Sarah at sarah@co.com for the report"     │
│ Output: {"name":"Sarah","email":"sarah@co.com"}          │
│                                                         │
│ NOW EXTRACT:                                            │
│ Input: "Ping Raj at raj@example.com and 9123456789"     │
└─────────────────────────────────────────────────────────┘
```

## 2.5 Python Example — Static Few-shot

In [ ]:
# FEW-SHOT: Hand-picked examples teach exact output format

few_shot_prompt = """
You extract product names and prices from customer messages.
Always respond in this exact JSON format: {"product": "...", "price": "..."}

Example 1:
Input: "I want to buy the Nike Air Max for 8500 rupees"
Output: {"product": "Nike Air Max", "price": "8500"}

Example 2:
Input: "Can I get the Sony headphones at 3200?"
Output: {"product": "Sony headphones", "price": "3200"}

Example 3:
Input: "Order the OnePlus Watch 2 for 14999"
Output: {"product": "OnePlus Watch 2", "price": "14999"}

Now extract:
Input: "I'd like the Bose QuietComfort 45 at 27000 please"
Output:
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[{"role": "user", "content": few_shot_prompt}]
)

output = response.content[0].text.strip()
data = json.loads(output)
print(data)  # → {"product": "Bose QuietComfort 45", "price": "27000"}

### 📋 Code Breakdown

| Concept | Explanation |
|---------|-------------|
| 3 curated examples | Each example teaches exact JSON format. Chosen to cover different phrasing patterns. |
| `"Now extract:"` separator | Clear boundary between examples and the real task — helps the model understand the pattern ends here |
| `"Output:"` at the end | Classic few-shot completion trigger: model continues the pattern |
| `json.loads(output)` | Since we enforced JSON format via examples, we can reliably parse the output |

## 2.6 Advanced Example — Dynamic Few-shot

> 🏭 **Production pattern:** Instead of using the same 3 examples for every query, pick the **most relevant examples** per query using semantic similarity. This is called **dynamic few-shot**.

In [ ]:
# DYNAMIC FEW-SHOT: Select the most relevant examples per query
# (Conceptual demo — uses simple keyword matching as a stand-in for embedding similarity)

# In production: use sentence-transformers embeddings + cosine similarity
# Here we demonstrate the concept with a simple keyword scorer

# --- Example Bank ---
example_bank = [
    {"input": "Nike Air Max for 8500 rupees",     "output": '{"product": "Nike Air Max", "price": "8500"}'},
    {"input": "Sony headphones at 3200",           "output": '{"product": "Sony headphones", "price": "3200"}'},
    {"input": "OnePlus Watch 2 for 14999",         "output": '{"product": "OnePlus Watch 2", "price": "14999"}'},
    {"input": "Apple AirPods Pro at 22000",        "output": '{"product": "Apple AirPods Pro", "price": "22000"}'},
    {"input": "Samsung Galaxy Buds costing 8999",  "output": '{"product": "Samsung Galaxy Buds", "price": "8999"}'},
]

def score_relevance(example, query):
    """Simple word-overlap scorer (production: use cosine similarity of embeddings)"""
    query_words = set(query.lower().split())
    example_words = set(example["input"].lower().split())
    return len(query_words & example_words)

def select_examples(query, bank, top_k=3):
    """Pick the top-k most relevant examples for this query"""
    scored = [(ex, score_relevance(ex, query)) for ex in bank]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [ex for ex, _ in scored[:top_k]]

def build_dynamic_few_shot_prompt(query, examples):
    prompt = "Extract the product name and price from the input message.\n"
    prompt += 'Always respond in JSON format: {"product": "...", "price": ""}\n\n'
    for i, ex in enumerate(examples, 1):
        prompt += f"Example {i}:\nInput: \"{ex['input']}\"\nOutput: {ex['output']}\n\n"
    prompt += f"Now extract:\nInput: \"{query}\"\nOutput:"
    return prompt

# --- Run it ---
user_query = "I want the Bose QuietComfort 45 headphones for 27000"

selected = select_examples(user_query, example_bank, top_k=3)
prompt = build_dynamic_few_shot_prompt(user_query, selected)

print("=== Selected examples ===")
for ex in selected:
    print(f"  - {ex['input']}")

response = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[{"role": "user", "content": prompt}]
)

result = json.loads(response.content[0].text.strip())
print("\n=== Extracted ===")
print(json.dumps(result, indent=2))

## 2.7 Learning Depth

### 🟢 Beginner View
Show the model a few examples of what you want, then give it the real task. It copies the pattern from the examples.

### 🟡 Intermediate View
**Example quality matters more than quantity.** 3 great examples beat 10 mediocre ones.  
- Examples should cover **edge cases** and vary in phrasing.
- Always produce the **exact output format** you need.
- **Order matters** — place the most typical example last (closest to the real task).

### 🔴 Production View
Teams maintain **"example banks"** — curated datasets of (input, ideal_output) pairs.  
Examples are selected dynamically using **semantic similarity** (embedding-based RAG-style lookup).  
This is called **dynamic few-shot** and dramatically improves accuracy over static examples.

---

## 2.8 Common Mistakes

| Mistake | Fix |
|---------|-----|
| ❌ Using bad/ambiguous examples | ✅ Each example must be unambiguous and correct |
| ❌ Too many examples (> 10) | ✅ Quality > quantity. 3–5 excellent examples is ideal |
| ❌ Static examples for all queries | ✅ Use dynamic selection for production |
| ❌ All examples are easy | ✅ Include edge cases in your bank |

---

## 2.9 Interview Questions

**Q1: What makes a curated example good vs bad in few-shot prompting?**  
A: A good example is unambiguous, covers an important input pattern, produces the exact desired output format, and differs from other examples. A bad example is vague, overlaps too much, or shows inconsistent output format.

**Q2: What is dynamic few-shot and why is it used in production?**  
A: Dynamic few-shot selects the most relevant examples per query using semantic similarity (embedding search). A fixed example set can't optimally cover all input variations — dynamic selection consistently picks the most helpful demonstrations.

**Q3: How does few-shot differ from fine-tuning?**  
A: Few-shot puts examples inside the prompt at inference time — no model weights change. Fine-tuning updates model weights using training data. Few-shot is faster and cheaper but limited by context window size. Fine-tuning permanently bakes in patterns without consuming prompt tokens.

---
# 3. COSTAR Framework
> **Number:** 03 | **Difficulty:** Beginner → Production

---

## 3.1 What Is It?

**COSTAR** is a structured prompt engineering framework. Instead of a raw instruction, you fill in **6 clearly defined components** that together give the LLM full clarity.

| Letter | Component | What It Defines |
|--------|-----------|------------------|
| **C** | Context | Background information about the situation |
| **O** | Objective | The specific goal of this prompt |
| **S** | Style | Writing format to follow |
| **T** | Tone | Emotional quality of the output |
| **A** | Audience | Who will read this output |
| **R** | Response | Format / length / structure of the output |

---

## 3.2 Why Do We Need It?

Without COSTAR, you write: *"Write a product description."*  
The model **guesses** style, tone, and audience → inconsistent outputs across agent runs.

COSTAR **eliminates ambiguity**. Each component removes one degree of freedom the model would otherwise guess randomly.

---

## 3.3 Real-world Analogy

> 📋 **Analogy:** A marketing director's creative brief to a copywriter.  
> They don't just say *"write an ad."* They specify: product, goal, style guide, tone of voice, target customer, and deliverable format.  
> **COSTAR is that creative brief — for LLMs.**

---

## 3.4 Agentic AI Context

COSTAR shines in agentic systems for:
- **Content generation agents** (blog writer, email drafter, report generator)
- **Multi-agent systems** where one agent specializes in writing
- **RAG systems** presenting content for a specific audience
- **Customer-facing agents** where tone consistency = brand safety

```
RAG Pipeline with COSTAR:

User Query → Retrieve Documents → Build COSTAR Prompt
                                          │
┌─────────────────────────────────────────▼──────────────────┐
│ C: We retrieved 3 docs about RBI repo rate changes         │
│ O: Summarize what this means for EMI borrowers             │
│ S: Simple prose, avoid tables                              │
│ T: Calm, informative, not alarming                         │
│ A: Middle-class Indian homeowners with home loans          │
│ R: 3 sentences max, avoid % figures, use plain language    │
└────────────────────────────────────────────────────────────┘
          │
          ▼
LLM Response perfectly tailored for the user
```

## 3.5 Python Example — COSTAR as a Reusable Function

In [ ]:
# COSTAR: Build structured system prompts as a reusable function

def build_costar_prompt(context, objective, style, tone, audience, response_format):
    """
    Build a COSTAR-structured system prompt.
    Each parameter maps directly to one COSTAR component.
    """
    return f"""
[CONTEXT]
{context}

[OBJECTIVE]
{objective}

[STYLE]
{style}

[TONE]
{tone}

[AUDIENCE]
{audience}

[RESPONSE FORMAT]
{response_format}
"""

# Build a system prompt for an edtech company in Andhra Pradesh
system_prompt = build_costar_prompt(
    context="We are an Andhra Pradesh-based edtech company offering competitive exam coaching.",
    objective="Write a WhatsApp message inviting students to a free demo class.",
    style="Conversational, short paragraphs, 1–2 emojis max.",
    tone="Encouraging, friendly, not pushy.",
    audience="Class 10–12 students preparing for NEET/JEE, Telugu-speaking background.",
    response_format="Under 80 words. End with a clear call-to-action link placeholder."
)

response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    system=system_prompt,         # COSTAR goes in the system prompt
    messages=[{"role": "user", "content": "Generate the message now."}]
)

print(response.content[0].text)

### 📋 Code Breakdown

| Concept | Explanation |
|---------|-------------|
| `build_costar_prompt()` function | Turns COSTAR into a **reusable builder** — standard pattern in agent codebases |
| `system=system_prompt` | COSTAR goes in the **system prompt** — it defines persistent behavior for the session |
| User message: `"Generate the message now."` | Just a trigger — all real instructions live in the system prompt via COSTAR |
| Localized audience param | Audience awareness ensures culturally appropriate output — critical for real-world products |

## 3.6 Advanced Example — Multi-tenant COSTAR (Dynamic Components)

> 🏭 **Production pattern:** In a multi-tenant product, you swap only the dynamic COSTAR components (C, O, A) per user while keeping S, T, R constant for brand consistency.

In [ ]:
# MULTI-TENANT COSTAR: Constant brand components + dynamic user-specific components

# Brand-level constants (set by product team, never change)
BRAND_STYLE = "Short sentences, active voice, no jargon."
BRAND_TONE = "Warm, professional, confidence-building."
BRAND_RESPONSE = "Under 100 words. Always end with a specific next step."

# User profiles (dynamic — changes per customer segment)
user_profiles = [
    {
        "segment": "Student (NEET prep)",
        "context": "EdTech platform for NEET exam preparation.",
        "audience": "18-year-old students from tier-2 cities, first-generation exam takers."
    },
    {
        "segment": "Corporate professional",
        "context": "B2B SaaS platform for HR teams.",
        "audience": "HR managers at mid-size companies, data-driven decision makers."
    }
]

objective = "Write an onboarding welcome message for a new user."

for profile in user_profiles:
    system = build_costar_prompt(
        context=profile["context"],
        objective=objective,
        style=BRAND_STYLE,         # constant
        tone=BRAND_TONE,           # constant
        audience=profile["audience"],
        response_format=BRAND_RESPONSE  # constant
    )
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system=system,
        messages=[{"role": "user", "content": "Generate the welcome message."}]
    )
    
    print(f"=== Segment: {profile['segment']} ===")
    print(response.content[0].text)
    print()

## 3.7 Learning Depth

### 🟢 Beginner View
COSTAR is a checklist for writing complete prompts. Fill in all 6 boxes and your prompt becomes much more specific → much better outputs.

### 🟡 Intermediate View
Not every task needs all 6 components.
- Data extraction task → Style/Tone/Audience matter less
- Content generation task → All 6 are critical
Learn which components drive quality for which task types.

### 🔴 Production View
In enterprise systems, COSTAR parameters are stored in a **configuration database per use-case**.  
When context changes (e.g., same agent serving English vs. Hindi speakers), only specific COSTAR components are swapped dynamically.  
This is a key pattern for **multi-tenant AI products**.

---

## 3.8 Common Mistakes

| Mistake | Fix |
|---------|-----|
| ❌ Conflicting components (Tone: formal + Style: use emojis) | ✅ All 6 components must be internally consistent |
| ❌ Vague audience: "General public" | ✅ Be specific: age, role, background, knowledge level |
| ❌ Over-specifying Response Format (10+ rules) | ✅ Keep Response Format to 2–3 clear directives |
| ❌ Using COSTAR for arithmetic calculations | ✅ Match tool to task complexity — COSTAR is for content tasks |

---

## 3.9 Interview Questions

**Q1: What problem does COSTAR solve that a simple instruction doesn't?**  
A: A simple instruction leaves the model to guess context, audience, style, and format — leading to inconsistent outputs. COSTAR eliminates ambiguity by explicitly specifying every dimension, producing consistent, on-brand, audience-appropriate responses.

**Q2: How would you implement COSTAR in a multi-agent system?**  
A: Each specialized agent gets its own COSTAR system prompt. The orchestrator passes task context dynamically into the C and O fields while S/T/A/R stay constant per agent.

**Q3: Is COSTAR a universal standard?**  
A: It's one structured approach among many (RTF: Role-Task-Format, APE: Action-Purpose-Expectation). What matters is the principle: be explicit about every dimension of what you need from the model.

---
# 4. Iterative Refinement Loop
> **Number:** 04 | **Difficulty:** Intermediate → Production

---

## 4.1 What Is It?

**Iterative refinement** is a process where an LLM/agent:
1. **Generates** an output
2. **Critiques** its own output (or has another model critique it)
3. **Refines** based on the critique
4. **Repeats** until a quality threshold is met

This is one of the most powerful patterns in modern Agentic AI.  
Instead of trusting a single LLM pass, you **build quality in through iteration**.

---

## 4.2 Why Do We Need It?

LLMs are probabilistic — their first output isn't always the best:
- First drafts are often shallow or miss requirements
- Complex tasks (code generation, research reports) require multiple passes
- Self-critique helps the model catch its own reasoning errors
- Loops allow progressive improvement toward a specification

---

## 4.3 Real-world Analogy

> 📝 **Analogy:** A journalist writes a first draft. An editor reviews it with comments. The journalist revises.  
> This cycle repeats 2–3 times before publication.  
> **Iterative refinement is this editor-writer feedback loop — automated inside an AI system.**

---

## 4.4 Agentic AI Context

This is the **heartbeat of agentic behavior**. Real agents don't just generate once:

| Agent Type | Refinement Loop |
|------------|------------------|
| Code agents | Generate code → run tests → see errors → fix → repeat |
| Research agents | Draft answer → check for missing sources → improve → verify |
| Writing agents | Generate draft → self-critique for clarity → rewrite weak sections |
| RAG systems | Generate answer → check if grounded in retrieved docs → refine if hallucinating |

```
ITERATIVE REFINEMENT LOOP IN AN AGENT:

  ┌──────────────────────────────────────────┐
  │                                          │
  │  ① GENERATE ──► First draft output       │
  │        │                                 │
  │        ▼                                 │
  │  ② CRITIQUE ──► Evaluate against rubric  │
  │        │        (checklist / score)      │
  │        ▼                                 │
  │  ③ CHECK ──────► Quality threshold met?  │
  │        │         YES ──► EXIT loop ✓     │
  │        │         NO  ──► Continue        │
  │        ▼                                 │
  │  ④ REFINE ─────► Improve based on        │
  │        │         critique feedback       │
  │        └──────────────────────────┐      │
  │  (Loop back to ② with new draft)  │      │
  └───────────────────────────────────┘      │
                                             │
  Max iterations reached? → EXIT anyway ────┘
```

## 4.5 Python Example — Complete Refinement Loop

In [ ]:
# ITERATIVE REFINEMENT LOOP: generate → critique → refine → repeat

def generate(task: str) -> str:
    """Step 1: Generate first draft"""
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        messages=[{"role": "user", "content": task}]
    )
    return response.content[0].text

def critique(draft: str, criteria: str) -> str:
    """Step 2: Evaluate the draft against explicit criteria"""
    prompt = f"""
You are a quality reviewer. Evaluate this draft against the criteria below.
List specific improvements needed.
If the draft already meets ALL criteria, write exactly: APPROVED

CRITERIA:
{criteria}

DRAFT:
{draft}

FEEDBACK:
"""
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

def refine(draft: str, feedback: str) -> str:
    """Step 3: Improve draft based on critique feedback"""
    prompt = f"""
Improve this draft based on the feedback provided. Keep what works, fix what doesn't.

ORIGINAL DRAFT:
{draft}

FEEDBACK:
{feedback}

IMPROVED VERSION:
"""
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


# ===== MAIN LOOP =====
task = "Write a 3-sentence product description for a solar-powered phone charger."

# Measurable criteria — vague criteria = useless critique
criteria = """
- Must mention eco-friendly benefit explicitly
- Must include a specific use case (e.g., hiking, travel, power outage)
- Must end with a call-to-action
- Must be under 60 words total
"""

MAX_ITERATIONS = 3  # ← CRITICAL: always set a hard cap

draft = generate(task)
print(f"📝 Draft 1:\n{draft}\n")
print("-" * 60)

for i in range(MAX_ITERATIONS):
    feedback = critique(draft, criteria)
    print(f"🔍 Feedback {i + 1}:\n{feedback}\n")
    
    if "APPROVED" in feedback:
        print("✅ Quality threshold met. Exiting loop.")
        break
    
    draft = refine(draft, feedback)
    print(f"📝 Draft {i + 2}:\n{draft}\n")
    print("-" * 60)
else:
    # Python for-else: runs only if loop completed WITHOUT a break
    print("⚠️  Max iterations reached. Using best available draft.")

print(f"\n🏁 FINAL OUTPUT:\n{draft}")

### 📋 Code Breakdown

| Concept | Explanation |
|---------|-------------|
| 3 separate functions | `generate / critique / refine` mirrors the real agentic pattern — each is a separate LLM call |
| `"APPROVED"` keyword | Simple exit condition — the critique model writes this when quality is sufficient |
| `MAX_ITERATIONS = 3` | **Critical safety mechanism** — without a max, the loop could run forever |
| `for...else` pattern | Python's `for-else` runs the `else` block only if the loop completed WITHOUT a `break` — elegant for "max reached" case |
| Criteria string | Explicit, measurable criteria make the critique step reliable. Vague criteria → vague feedback → no improvement |

## 4.6 Production Pattern — Refinement with Score-based Exit

> 🏭 **Advanced pattern:** Instead of a keyword exit, use a **numeric quality score** for more granular control.

In [ ]:
# SCORED REFINEMENT: Numeric quality score as exit condition
# More production-grade than a keyword check

def score_draft(draft: str, criteria: str) -> dict:
    """
    Returns a score (0-10) and specific feedback.
    Score >= 8 = approved.
    """
    prompt = f"""
You are a quality reviewer. Evaluate this draft against the criteria.

CRITERIA:
{criteria}

DRAFT:
{draft}

Respond ONLY in this JSON format:
{{"score": 0, "issues": ["issue1", "issue2"], "suggestions": ["suggestion1"]}}

Score 0-10 where 10 = perfect. List specific issues and suggestions.
"""
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.content[0].text.strip())


task = "Write a 3-sentence product description for a solar-powered phone charger."
criteria = """
- Must mention eco-friendly benefit
- Must include a specific use case
- Must end with a call-to-action
- Must be under 60 words
"""

MAX_ITERATIONS = 4
QUALITY_THRESHOLD = 8  # Score must be >= 8 to exit

draft = generate(task)
api_calls = 1  # Track cost

for i in range(MAX_ITERATIONS):
    evaluation = score_draft(draft, criteria)
    api_calls += 1
    
    print(f"Iteration {i + 1} | Score: {evaluation['score']}/10")
    print(f"Issues: {evaluation['issues']}")
    
    if evaluation['score'] >= QUALITY_THRESHOLD:
        print(f"\n✅ Score {evaluation['score']} >= threshold {QUALITY_THRESHOLD}. Done!")
        break
    
    # Refine using suggestions
    feedback_str = "\n".join(evaluation['suggestions'])
    draft = refine(draft, feedback_str)
    api_calls += 1
    print()
else:
    print(f"\n⚠️  Max iterations reached (score={evaluation['score']}).")

print(f"\nTotal API calls used: {api_calls}")
print(f"\n🏁 FINAL:\n{draft}")

## 4.7 Learning Depth

### 🟢 Beginner View
Generate a first answer, ask the same model "what's wrong with this?", ask it to fix those problems. Repeat 2–3 times. Your final output will be much better than the first.

### 🟡 Intermediate View
The **critique step is the most important**. If your rubric is vague, the critique is useless.  
Use a separate LLM call with an explicit evaluation checklist.  
You can also use **two different models**: a smaller/faster model for generation + a stronger model for critique.

### 🔴 Production View
In **LangGraph**: conditional edges route between generate→critique→refine nodes based on quality score.  
In **AutoGen**: Writer + Critic agents in a `GroupChat` with `MAX_ROUNDS` parameter.  
Cost control is critical — log API calls per loop and set hard budget limits per user request.

---

## 4.8 Production Framework Mapping

| Framework | How Refinement Loop Is Implemented |
|-----------|-------------------------------------|
| LangGraph | Conditional edges route between generate→critique→refine nodes |
| AutoGen | Writer + Critic agents in GroupChat with MAX_ROUNDS |
| CrewAI | Task with reviewer agent checking output quality |
| OpenAI Agents SDK | Agent loops with output schema validation as exit condition |
| Custom Python | Simple while/for loop — most transparent and controllable |

---

## 4.9 Common Mistakes

| Mistake | Fix |
|---------|-----|
| ❌ No max iterations | ✅ **Always** set `MAX_ITERATIONS`. This is the #1 mistake. |
| ❌ Vague exit condition | ✅ Define measurable criteria — keyword, score threshold, or Pydantic validation |
| ❌ Not tracking API cost | ✅ Log calls per loop. 3 iterations × 3 calls = 9 API calls per user request |
| ❌ Conflicting criteria | ✅ Validate criteria consistency before the loop starts |
| ❌ Over-refining | ✅ Diminishing returns after 3–4 iterations is common. Know when to stop. |

---

## 4.10 Interview Questions

**Q1: What is the iterative refinement loop and why is it considered "agentic" behavior?**  
A: It's a generate→critique→refine cycle where an agent improves its own output. It's agentic because it involves autonomous goal-directed behavior — the agent keeps working toward a quality target without human input at each step.

**Q2: How do you prevent an iterative loop from running indefinitely?**  
A: Set a `MAX_ITERATIONS` constant. Define a clear, checkable exit condition (keyword, score ≥ threshold, Pydantic validation pass). In LangGraph, use a conditional edge that routes to `END` after N iterations regardless of quality.

**Q3: What's the difference between self-refinement and multi-agent refinement?**  
A: In self-refinement, the same model generates and critiques (cheaper, simpler). In multi-agent refinement, a separate critic agent reviews the output. Production systems often use a weaker model for generation and a stronger one for critique to balance cost and quality.

---
# 5. Big Picture Summary

---

## 5.1 How All 4 Techniques Connect

```
COMPLEXITY  ──────────────────────────────────────────────►

Simple          Medium           Complex         Very Complex
   │               │                │                 │
   ▼               ▼                ▼                 ▼
Zero-shot      Few-shot          COSTAR         COSTAR +
(Direct ask)  (Show examples)  (Full context)   Iteration loop
```

**These are NOT mutually exclusive!** Production systems combine all 4:
- COSTAR defines the agent persona (system prompt)
- Few-shot examples demonstrate the exact output format
- Zero-shot handles quick routing decisions
- Iteration loop polishes complex outputs

---

## 5.2 Agentic AI System Connection

| System | Zero-shot | Few-shot | COSTAR | Refinement |
|--------|-----------|----------|--------|------------|
| **AI Agents** | Intent routing | Tool call formatting | Agent persona | Multi-step task completion |
| **Multi-Agent Systems** | Quick decisions | Shared prompt library | Per-agent system prompts | Critic agent loop |
| **RAG Applications** | Query rewriting | Citation format | Audience-aware answers | Hallucination check |
| **Tool Calling Systems** | Tool selection | JSON format demo | Tool context | Multi-step planning |
| **Workflow Automation** | Classification | Pattern matching | Persona per workflow | Quality assurance |

---

## 5.3 Industry Framework Mapping

| Framework | Zero-shot | Few-shot | COSTAR | Refinement |
|-----------|-----------|----------|--------|------------|
| OpenAI Agents SDK | Intent router | Tool call examples | Agent instructions | Agent loop |
| LangGraph | Node LLM calls | Prompt templates | System prompts | Conditional edges |
| CrewAI | Task descriptions | Role examples | Agent backstory | Reviewer crew |
| AutoGen | Quick queries | Conversation seeding | Agent system msg | GroupChat rounds |
| PydanticAI | Simple queries | Output validators | System prompts | Retry on validation fail |
| MCP | Tool descriptions | Tool use examples | Server context | Multi-turn calls |

---

## 5.4 What to Learn Next

| Next Topic | Why It Builds on This Module |
|------------|------------------------------|
| Chain-of-Thought Prompting | Supercharges zero-shot for reasoning: "Think step by step" |
| Prompt Templates (LangChain Hub) | Production management of few-shot banks and COSTAR templates |
| Structured Output / JSON Mode | Enforces format at API level — eliminates parsing failures |
| LangGraph State Machines | Implements refinement loops as proper graphs with state tracking |
| Prompt Evaluation (Evals) | Measuring whether your prompts actually perform at scale |
| RAG (Retrieval Augmented Generation) | Connects all 4 techniques into a real information pipeline |

---
# 6. Knowledge Check
### 10 Practical Questions

> ✏️ **Instructions:** Try to answer each question before revealing the answer.  
> Run each cell to see the model answer.

---

**Q1:** You need to classify customer complaints into 5 categories. You have no labeled examples. Which technique do you start with, and how do you maximize its performance?

**Q2:** Your agent's tool-call JSON format is inconsistent. Which technique best solves this, and why?

**Q3:** You're building a multi-tenant content generation agent serving both formal enterprise clients and casual consumer users. How do you use COSTAR to handle both without two separate codebases?

**Q4:** Describe the exact flow of an iterative refinement loop for a code generation agent. What happens at each step?

**Q5:** What is dynamic few-shot? How is it different from static few-shot? When would you use it in production?

**Q6:** Your refinement loop is running 8+ iterations and costs are high. What are 3 things you would do to fix this?

**Q7:** A colleague says: "Just fine-tune the model instead of using few-shot." When would they be right, and when would few-shot still be the better choice?

**Q8:** In COSTAR, what does the "Audience" component specifically change about the LLM's output? Give an example where changing only the Audience field produces a very different result.

**Q9:** You're implementing an iterative loop in LangGraph. How do you structure the graph to implement generate → critique → conditional refinement?

**Q10:** Combine all 4 techniques: design a prompt pipeline for a blog-writing agent that uses all 4 techniques you learned.

In [ ]:
# === KNOWLEDGE CHECK ANSWERS ===
# Run this cell to print all model answers.
# Try answering each question yourself FIRST before running this.

answers = {
    "Q1": """Start with zero-shot. Maximize it by: (1) adding a system role ('You are a senior customer experience classifier'), 
(2) listing all 5 categories with brief descriptions, (3) specifying output format ('Respond with ONLY the category name'), 
(4) adding 'Think step by step'. If accuracy is below target after testing, upgrade to few-shot.""",
    
    "Q2": """Few-shot prompting. Show the model 3–5 examples of the exact JSON tool call format you expect. 
The model learns the pattern precisely from examples, ensuring consistent format every time. 
Optionally combine with structured output (JSON mode) for guaranteed parsing.""",
    
    "Q3": """Store S (Style), T (Tone), R (Response format) as constants in code (brand-level). 
Store C (Context) and A (Audience) in a per-tenant config database. 
O (Objective) is dynamic per request. At runtime, compose the COSTAR prompt by swapping only the tenant-specific fields.""",
    
    "Q4": """1. GENERATE: Agent calls LLM to write code for the task. 
2. EXECUTE: Run the code in a sandbox, capture output/errors. 
3. CRITIQUE: Feed errors back to LLM as feedback. 
4. CHECK: If tests pass → exit loop. If tests fail → continue. 
5. REFINE: LLM rewrites the code incorporating the error feedback. 
Repeat with MAX_ITERATIONS cap (typically 5).""",
    
    "Q5": """Static few-shot: same fixed examples for every query. 
Dynamic few-shot: embeds each query, does semantic similarity search against an example bank, selects top-K most relevant examples per query. 
Use in production when: you have a large, diverse input space, your static examples cover only some patterns, and accuracy on edge cases needs improvement.""",
    
    "Q6": """1. Tighten the exit criteria — make them more achievable (score threshold from 9 to 7.5). 
2. Improve the generate step prompt using COSTAR/few-shot so first drafts are higher quality. 
3. Reduce MAX_ITERATIONS from 8 to 3 and implement hard budget cutoff (e.g., max $0.05 per request). 
Also: log which criteria keep failing to identify systematic prompt improvements.""",
    
    "Q7": """Fine-tuning is better when: you have 1000+ labeled examples, the task is very domain-specific, 
you need consistent performance across millions of requests (lower inference cost), or the pattern is too complex for in-context learning. 
Few-shot is better when: you have few examples, the task changes frequently, you need fast iteration, or cost of fine-tuning job is not justified.""",
    
    "Q8": """Audience changes vocabulary, complexity, assumed knowledge, and cultural references. 
Example: Same objective 'Explain compound interest'. 
Audience A: 'PhD economists' → uses PV/FV formulas, continuous compounding notation. 
Audience B: 'Village shopkeeper with Class 8 education' → uses a grocery savings analogy, no formulas, simple Telugu-style phrasing.""",
    
    "Q9": """Create 3 nodes: generate_node, critique_node, refine_node. 
Connect: generate_node → critique_node. 
Add conditional edge from critique_node: if score >= threshold → END, else → refine_node. 
Connect: refine_node → critique_node. 
Store iteration_count in state. Add iteration check: if count >= MAX_ITER → END regardless of score.""",
    
    "Q10": """Pipeline design for blog-writing agent:
1. COSTAR system prompt: defines the blog's target audience, style guide, tone, and response format.
2. Few-shot examples: 2 example (topic → blog outline) pairs to show the structural format expected.
3. Zero-shot: within the agent, intent routing classifies the user's topic into a blog category.
4. Iterative loop: generate draft → critique against SEO/clarity rubric → refine → repeat max 3 times.
Result: consistent, high-quality, audience-targeted blog posts every time."""
}

for q, a in answers.items():
    print(f"{'='*60}")
    print(f"  {q}")
    print(f"{'='*60}")
    print(f"  {a}")
    print()

---
# 7. Mini Projects
### Build to reinforce what you learned

---

## Project 1 — Smart Customer Support Classifier

**Concepts used:** Zero-shot → Few-shot upgrade + COSTAR  
**What you build:** A classifier that routes support tickets, then upgrades to few-shot when accuracy drops

**Steps:**
1. Build a zero-shot classifier for 5 categories: billing, technical, shipping, returns, other
2. Test it on 20 sample tickets, measure accuracy
3. Identify failure categories, add 3 curated few-shot examples per failure
4. Re-test and compare accuracy improvement
5. Add a COSTAR system prompt to make the agent explain its classification reasoning

---

## Project 2 — Self-Improving Blog Writer Agent

**Concepts used:** COSTAR + Iterative Refinement Loop  
**What you build:** A blog writing agent that automatically improves its own drafts

**Steps:**
1. Design a COSTAR system prompt for a tech blog targeting Indian developers
2. Build `generate()` to draft a 300-word post
3. Build `critique()` with a rubric: readability, examples, CTA, SEO keywords
4. Build `refine()` to apply feedback
5. Implement the loop with MAX_ITERATIONS=3 and APPROVED exit condition
6. Print each draft to observe the quality improvement

---

## Project 3 — Dynamic Few-shot RAG Answer Generator

**Concepts used:** Few-shot (dynamic) + COSTAR + Zero-shot  
**What you build:** A RAG-style system where examples are selected per query

**Steps:**
1. Create a bank of 20 (question, ideal_answer) pairs about a topic of your choice
2. Embed all questions using `sentence-transformers`
3. At query time, embed the user's question and find top-3 similar examples
4. Inject dynamic examples into a few-shot prompt
5. Use COSTAR to define the answer format for a specific audience
6. Compare dynamic vs. static few-shot quality on 10 test questions

## Project 1 Starter Code — Support Classifier

In [ ]:
# PROJECT 1 STARTER: Support ticket classifier with zero-shot → few-shot upgrade

CATEGORIES = ["billing", "technical", "shipping", "returns", "other"]

# --- ZERO-SHOT VERSION ---
def classify_zero_shot(ticket: str) -> str:
    prompt = f"""
Classify this customer support ticket into exactly one of these categories:
billing, technical, shipping, returns, other

Ticket: "{ticket}"

Respond with ONLY the category name, nothing else.
"""
    response = client.messages.create(
        model=MODEL,
        max_tokens=20,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip().lower()


# --- FEW-SHOT VERSION ---
FEW_SHOT_EXAMPLES = [
    ("My invoice shows wrong amount charged", "billing"),
    ("App keeps crashing on my Android phone", "technical"),
    ("Package not delivered after 10 days", "shipping"),
    ("I want to return the damaged product", "returns"),
    ("What are your working hours?", "other"),
]

def classify_few_shot(ticket: str) -> str:
    examples_text = "\n".join(
        [f'Ticket: "{t}" → Category: {c}' for t, c in FEW_SHOT_EXAMPLES]
    )
    prompt = f"""
Classify customer support tickets into: billing, technical, shipping, returns, other

{examples_text}

Ticket: "{ticket}" → Category:
"""
    response = client.messages.create(
        model=MODEL,
        max_tokens=20,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip().lower()


# --- TEST SET ---
test_tickets = [
    ("Double charged for last month's subscription", "billing"),
    ("Login button not working on website", "technical"),
    ("Where is my order #A1234?", "shipping"),
    ("Product broke after 2 days, want refund", "returns"),
    ("Do you have a referral program?", "other"),
]

print("=== Zero-shot vs Few-shot Comparison ===")
print(f"{'Ticket':<45} {'Expected':<12} {'Zero-shot':<12} {'Few-shot':<12}")
print("-" * 85)

for ticket, expected in test_tickets:
    zs = classify_zero_shot(ticket)
    fs = classify_few_shot(ticket)
    zs_mark = "✅" if zs == expected else "❌"
    fs_mark = "✅" if fs == expected else "❌"
    print(f"{ticket[:43]:<45} {expected:<12} {zs+' '+zs_mark:<12} {fs+' '+fs_mark:<12}")

## Project 2 Starter Code — Blog Writer with Refinement

In [ ]:
# PROJECT 2 STARTER: COSTAR + Iterative refinement for blog writing

# Step 1: COSTAR system prompt for the blog agent
blog_system = build_costar_prompt(
    context="Tech blog targeting Indian software developers and aspiring AI engineers.",
    objective="Write an engaging blog post on the given topic.",
    style="Conversational yet technical. Use subheadings, short paragraphs, 1 code analogy.",
    tone="Friendly, encouraging, mentor-like. Avoid corporate jargon.",
    audience="Indian developers aged 22–30, comfortable with Python, new to AI concepts.",
    response_format="300 words. H2 subheadings. End with a 1-sentence key takeaway."
)

# Step 2: Blog criteria for the critic
blog_criteria = """
- Includes at least one real-world analogy
- Has at least 2 subheadings
- Ends with a clear 1-sentence key takeaway
- Uses simple English (avoid words like 'utilize', 'leverage', 'paradigm')
- Between 280 and 320 words
"""

topic = "Why Zero-shot Prompting is the starting point for every AI Engineer"

# Step 3: Generate first draft using COSTAR
def generate_blog(topic: str, system: str) -> str:
    response = client.messages.create(
        model=MODEL,
        max_tokens=600,
        system=system,
        messages=[{"role": "user", "content": f"Write a blog post about: {topic}"}]
    )
    return response.content[0].text

# Step 4: Run the refinement loop
MAX_ITER = 3
draft = generate_blog(topic, blog_system)
word_count = len(draft.split())
print(f"📝 Draft 1 ({word_count} words):\n{draft[:300]}...\n")

for i in range(MAX_ITER):
    feedback = critique(draft, blog_criteria)
    print(f"🔍 Feedback {i+1}: {feedback[:200]}...\n")
    
    if "APPROVED" in feedback:
        print("✅ Blog approved!")
        break
    
    draft = refine(draft, feedback)
    word_count = len(draft.split())
    print(f"📝 Draft {i+2} ({word_count} words): ready.\n")
else:
    print("⚠️ Max iterations reached.")

print("\n" + "="*60)
print("🏁 FINAL BLOG POST:")
print("="*60)
print(draft)

---
# ⚡ Quick Reference Card
### Pin this cell for fast revision

---

| Technique | When to Use | Key Rule | Production Pattern |
|-----------|-------------|----------|--------------------|
| **Zero-shot** | Simple, well-known tasks; rapid prototyping | Add role + explicit output format | Version-controlled prompt files + evals |
| **Few-shot** | Need consistent format; domain-specific patterns | Curate quality > quantity; 3–5 examples | Dynamic example banks with embedding search |
| **COSTAR** | Content generation; multi-tenant agents; brand-safe output | All 6 components must be internally consistent | Config DB per tenant; dynamic C/O, constant S/T/R |
| **Iterative Refinement** | Complex output; quality-critical; multi-step tasks | ALWAYS set MAX_ITERATIONS; use measurable exit criteria | LangGraph conditional edges; cost monitoring per loop |

---

### 🔑 The Golden Rules

1. **Zero-shot fails silently** — always test on edge cases before deploying
2. **Few-shot quality > quantity** — 3 great examples beat 10 mediocre ones
3. **COSTAR = creative brief** — fill every field, keep them consistent
4. **Refinement without MAX_ITERATIONS = infinite loop risk** — always cap it
5. **In production, you combine all 4** — not choose one

---

*Notebook generated from Module 3.4 — Agentic AI Engineer Roadmap (Balaji Chippada)*  
*Mentor: Senior Agent Engineer & AI Architect | Personal Study Reference*